Григорьева Анастасия 3391 (Домашнее задание №4)

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import requests
import zipfile
import io
import os

url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"

# Используем requests для скачивания
response = requests.get(url)
with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
    # Распаковываем основной архив
    zip_ref.extractall("bank_data")

# Распаковываем вложенный архив bank.zip
with zipfile.ZipFile("bank_data/bank.zip") as inner_zip:
    inner_zip.extractall("bank_data")

# Загрузка данных
df = pd.read_csv('bank_data/bank-full.csv', sep=';')

print(f"Данные загружены. Размер: {df.shape}")

# Выбор нужных столбцов
columns = ['age', 'job', 'marital', 'education', 'balance', 'housing',
           'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
           'previous', 'poutcome', 'y']
df = df[columns]

# Проверка пропущенных значений
print("Пропущенные значения:")
print(df.isnull().sum())

# Вопрос 1: Самое частое значение education
print("\nВопрос 1:")
education_mode = df['education'].mode()[0]
print(f"Самое частое значение education: {education_mode}")

# Вопрос 2: Корреляционная матрица
print("\nВопрос 2:")
numeric_columns = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
correlation_matrix = df[numeric_columns].corr()

# Находим пару с максимальной корреляцией (исключая диагональ)
max_corr = 0
pair = ()
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr = abs(correlation_matrix.iloc[i, j])
        if corr > max_corr:
            max_corr = corr
            pair = (correlation_matrix.columns[i], correlation_matrix.columns[j])

print(f"Наибольшая корреляция: {pair[0]} и {pair[1]} ({max_corr:.3f})")

# Кодирование целевой переменной
df['y'] = df['y'].map({'yes': 1, 'no': 0})

# Разделение данных
X = df.drop('y', axis=1)
y = df['y']

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp  # 0.25 * 0.8 = 0.2
)

print(f"\nРазмеры наборов данных:")
print(f"Train: {X_train.shape[0]}")
print(f"Validation: {X_val.shape[0]}")
print(f"Test: {X_test.shape[0]}")

# Вопрос 3: Взаимная информация
print("\nВопрос 3:")
categorical_columns = ['job', 'marital', 'education', 'housing', 'contact', 'month', 'poutcome']

# Создаем копию данных для вычисления mutual information
X_train_categorical = X_train[categorical_columns].copy()

# Кодируем категориальные переменные для mutual_info_classif
le = LabelEncoder()
X_train_encoded = X_train_categorical.apply(lambda x: le.fit_transform(x.astype(str)))

mi_scores = mutual_info_classif(X_train_encoded, y_train, random_state=42)
mi_results = pd.DataFrame({
    'feature': categorical_columns,
    'mi_score': [round(score, 2) for score in mi_scores]
})
mi_results = mi_results.sort_values('mi_score', ascending=False)

print("Взаимная информация с категориальными переменными:")
print(mi_results)

max_mi_feature = mi_results.iloc[0]['feature']
print(f"Переменная с наибольшей взаимной информацией: {max_mi_feature}")

# Подготовка данных для логистической регрессии (one-hot encoding)
X_train_encoded = pd.get_dummies(X_train, columns=categorical_columns, drop_first=False)
X_val_encoded = pd.get_dummies(X_val, columns=categorical_columns, drop_first=False)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_columns, drop_first=False)

# Выравнивание столбцов (на случай разных категорий в train/val/test)
common_columns = X_train_encoded.columns.intersection(X_val_encoded.columns).intersection(X_test_encoded.columns)
X_train_encoded = X_train_encoded[common_columns]
X_val_encoded = X_val_encoded[common_columns]
X_test_encoded = X_test_encoded[common_columns]

# Вопрос 4: Логистическая регрессия
print("\nВопрос 4:")
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_encoded, y_train)

y_val_pred = model.predict(X_val_encoded)
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f"Точность на валидационном наборе: {val_accuracy:.2f}")

# Вопрос 5: Feature elimination
print("\nВопрос 5:")
base_accuracy = val_accuracy

features_to_test = ['age', 'balance', 'marital', 'previous']
accuracy_differences = {}

for feature in features_to_test:
    # Создаем копии данных без одного признака
    X_train_reduced = X_train.drop(feature, axis=1)
    X_val_reduced = X_val.drop(feature, axis=1)

    # One-hot кодирование
    X_train_reduced_encoded = pd.get_dummies(X_train_reduced, columns=[col for col in categorical_columns if col != feature], drop_first=False)
    X_val_reduced_encoded = pd.get_dummies(X_val_reduced, columns=[col for col in categorical_columns if col != feature], drop_first=False)

    # Выравнивание столбцов
    common_columns_reduced = X_train_reduced_encoded.columns.intersection(X_val_reduced_encoded.columns)
    X_train_reduced_encoded = X_train_reduced_encoded[common_columns_reduced]
    X_val_reduced_encoded = X_val_reduced_encoded[common_columns_reduced]

    # Обучение модели
    model_reduced = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_reduced.fit(X_train_reduced_encoded, y_train)

    # Предсказание и оценка
    y_val_pred_reduced = model_reduced.predict(X_val_reduced_encoded)
    reduced_accuracy = accuracy_score(y_val, y_val_pred_reduced)

    # Разница в точности
    difference = base_accuracy - reduced_accuracy
    accuracy_differences[feature] = abs(difference)
    print(f"Без {feature}: точность = {reduced_accuracy:.3f}, разница = {difference:.3f}")

min_diff_feature = min(accuracy_differences, key=accuracy_differences.get)
print(f"Признак с наименьшей разницей в точности: {min_diff_feature}")

# Вопрос 6: Регуляризация
print("\nВопрос 6:")
C_values = [0.01, 0.1, 1, 10]
best_accuracy = 0
best_C = None

for C in C_values:
    model_reg = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model_reg.fit(X_train_encoded, y_train)

    y_val_pred_reg = model_reg.predict(X_val_encoded)
    val_accuracy_reg = accuracy_score(y_val, y_val_pred_reg)

    print(f"C = {C}: точность = {val_accuracy_reg:.3f}")

    if val_accuracy_reg > best_accuracy:
        best_accuracy = val_accuracy_reg
        best_C = C

print(f"Лучшее значение C: {best_C} с точностью {best_accuracy:.3f}")


Данные загружены. Размер: (45211, 17)
Пропущенные значения:
age          0
job          0
marital      0
education    0
balance      0
housing      0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

Вопрос 1:
Самое частое значение education: secondary

Вопрос 2:
Наибольшая корреляция: pdays и previous (0.455)

Размеры наборов данных:
Train: 27126
Validation: 9042
Test: 9043

Вопрос 3:
Взаимная информация с категориальными переменными:
     feature  mi_score
6   poutcome      0.03
5      month      0.02
0        job      0.01
4    contact      0.01
3    housing      0.01
2  education      0.00
1    marital      0.00
Переменная с наибольшей взаимной информацией: poutcome

Вопрос 4:
Точность на валидационном наборе: 0.90

Вопрос 5:
Без age: точность = 0.903, разница = -0.000
Без balance: точность = 0.903, разница = -0.000
Без marital: точность = 0.902, разница = 0.001
Без previous: точность

1. Вопрос: secondary.
2. Вопрос: pdays и previous.
3. Вопрос: poutcome.
4. Вопрос: 0.9.
5. Вопрос: previous.
6. Вопрос: 10.




